In [1]:
# Import SparkSession, which is the main entry point for working with Spark.
# It allows us to create DataFrames, run Spark SQL, and manage Spark jobs.
from pyspark.sql import SparkSession

# Import built-in PySpark functions and give them the shorter alias "F".
# We will later use functions such as F.col(), F.count(), F.avg(), and F.to_date().
from pyspark.sql import functions as F


# Create or reuse a SparkSession.
# appName gives this Spark application a readable name that can appear in the Spark UI.
# getOrCreate() will use an existing SparkSession if one already exists,
# otherwise it will create a new one.
spark = (
    SparkSession.builder
    .appName("CS675SupplyChainProject")
    .getOrCreate()
)

# Print the Spark version so we can document the environment used for the project.
print("Spark version:", spark.version)

Spark version: 4.1.2


In [2]:
# Path to the supply chain CSV inside the Docker container.
# The professor's Docker setup mounts the local work folder to /home/jovyan/work.
inventory_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/supply_chain_dataset1.csv"

# Read the CSV file into a Spark DataFrame.
inventory_df = (
    spark.read
    .option("header", True)       # First row contains column names.
    .option("inferSchema", True)  # Let Spark infer data types during exploration.
    .csv(inventory_path)
)

print("Inventory dataset loaded successfully.")

Inventory dataset loaded successfully.


In [3]:
# Display the schema so we can see each column name and Spark data type.
inventory_df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- SKU_ID: string (nullable = true)
 |-- Warehouse_ID: string (nullable = true)
 |-- Supplier_ID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Units_Sold: integer (nullable = true)
 |-- Inventory_Level: integer (nullable = true)
 |-- Supplier_Lead_Time_Days: integer (nullable = true)
 |-- Reorder_Point: integer (nullable = true)
 |-- Order_Quantity: integer (nullable = true)
 |-- Unit_Cost: double (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Promotion_Flag: integer (nullable = true)
 |-- Stockout_Flag: integer (nullable = true)
 |-- Demand_Forecast: double (nullable = true)



In [4]:
# Display the first 10 records so we can understand what one row represents.
# truncate=False shows the complete values without shortening them.
inventory_df.show(10, truncate=False)

+----------+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|Date      |SKU_ID|Warehouse_ID|Supplier_ID|Region|Units_Sold|Inventory_Level|Supplier_Lead_Time_Days|Reorder_Point|Order_Quantity|Unit_Cost|Unit_Price|Promotion_Flag|Stockout_Flag|Demand_Forecast|
+----------+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|2024-01-01|SKU_1 |WH_1        |SUP_8      |West  |10        |592            |14                     |379          |0             |13.95    |20.48     |0             |0            |8.52           |
|2024-01-02|SKU_1 |WH_1        |SUP_8      |West  |17        |575            |14                     |379          |0             |13.95    |20.48     |0             |0            |18.63          |
|2024-01-0

In [5]:
# Count the total number of rows in the dataset.
# count() is a Spark action because it requires Spark to process the data.
inventory_rows = inventory_df.count()

# Count the number of columns using the DataFrame schema metadata.
inventory_columns = len(inventory_df.columns)

print("Rows:", inventory_rows)
print("Columns:", inventory_columns)

Rows: 91250
Columns: 15


In [6]:
# Check whether Date + SKU_ID + Warehouse_ID uniquely identifies each row.
# If every combination appears exactly once, this is likely the grain of the dataset.

grain_check_df = (
    inventory_df
    .groupBy("Date", "SKU_ID", "Warehouse_ID")
    .count()
    .filter(F.col("count") > 1)
)

grain_check_df.show(20, truncate=False)

+----+------+------------+-----+
|Date|SKU_ID|Warehouse_ID|count|
+----+------+------------+-----+
+----+------+------------+-----+



In [7]:
# Count the number of unique values in important categorical dimensions.

inventory_df.select(
    F.countDistinct("Date").alias("Unique_Dates"),
    F.countDistinct("SKU_ID").alias("Unique_SKUs"),
    F.countDistinct("Warehouse_ID").alias("Unique_Warehouses"),
    F.countDistinct("Supplier_ID").alias("Unique_Suppliers"),
    F.countDistinct("Region").alias("Unique_Regions")
).show()

+------------+-----------+-----------------+----------------+--------------+
|Unique_Dates|Unique_SKUs|Unique_Warehouses|Unique_Suppliers|Unique_Regions|
+------------+-----------+-----------------+----------------+--------------+
|         365|         50|                5|              10|             4|
+------------+-----------+-----------------+----------------+--------------+



In [8]:
# Check whether the dataset forms a complete Date × SKU × Warehouse structure.

expected_rows = (
    inventory_df.select("Date").distinct().count()
    * inventory_df.select("SKU_ID").distinct().count()
    * inventory_df.select("Warehouse_ID").distinct().count()
)

actual_rows = inventory_df.count()

print("Expected Date × SKU × Warehouse combinations:", expected_rows)
print("Actual rows:", actual_rows)

Expected Date × SKU × Warehouse combinations: 91250
Actual rows: 91250


In [9]:
# Check the earliest and latest dates in the dataset.
# This helps confirm the analysis period and whether external datasets
# such as holidays overlap with the same dates.

inventory_df.select(
    F.min("Date").alias("Start_Date"),
    F.max("Date").alias("End_Date")
).show()

+----------+----------+
|Start_Date|  End_Date|
+----------+----------+
|2024-01-01|2024-12-30|
+----------+----------+



In [10]:
# Count missing values in every column.
# isNull() returns True for missing values.
# Casting True/False to integer converts them to 1/0, allowing us to sum them.

missing_values_df = inventory_df.select(
    [
        F.sum(
            F.col(column_name).isNull().cast("int")
        ).alias(column_name)
        for column_name in inventory_df.columns
    ]
)

missing_values_df.show(truncate=False)

+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|Date|SKU_ID|Warehouse_ID|Supplier_ID|Region|Units_Sold|Inventory_Level|Supplier_Lead_Time_Days|Reorder_Point|Order_Quantity|Unit_Cost|Unit_Price|Promotion_Flag|Stockout_Flag|Demand_Forecast|
+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|0   |0     |0           |0          |0     |0         |0              |0                      |0            |0             |0        |0         |0             |0            |0              |
+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+



In [11]:
# Generate descriptive statistics for numeric columns.
# This helps identify unusual minimums, maximums, and variation
# before any preprocessing is performed.

inventory_df.describe().show(truncate=False)

+-------+------+------------+-----------+------+------------------+------------------+-----------------------+------------------+-----------------+------------------+------------------+-------------------+-------------+------------------+
|summary|SKU_ID|Warehouse_ID|Supplier_ID|Region|Units_Sold        |Inventory_Level   |Supplier_Lead_Time_Days|Reorder_Point     |Order_Quantity   |Unit_Cost         |Unit_Price        |Promotion_Flag     |Stockout_Flag|Demand_Forecast   |
+-------+------+------------+-----------+------+------------------+------------------+-----------------------+------------------+-----------------+------------------+------------------+-------------------+-------------+------------------+
|count  |91250 |91250       |91250      |91250 |91250             |91250             |91250                  |91250             |91250            |91250             |91250             |91250              |91250        |91250             |
|mean   |NULL  |NULL        |NULL       |NUL

In [12]:
# Check minimum and maximum values for the most important numeric fields.

inventory_df.select(
    F.min("Units_Sold").alias("Min_Units_Sold"),
    F.max("Units_Sold").alias("Max_Units_Sold"),
    F.min("Inventory_Level").alias("Min_Inventory"),
    F.max("Inventory_Level").alias("Max_Inventory"),
    F.min("Supplier_Lead_Time_Days").alias("Min_Lead_Time"),
    F.max("Supplier_Lead_Time_Days").alias("Max_Lead_Time"),
    F.min("Unit_Cost").alias("Min_Unit_Cost"),
    F.max("Unit_Cost").alias("Max_Unit_Cost"),
    F.min("Unit_Price").alias("Min_Unit_Price"),
    F.max("Unit_Price").alias("Max_Unit_Price"),
    F.min("Demand_Forecast").alias("Min_Forecast"),
    F.max("Demand_Forecast").alias("Max_Forecast")
).show(truncate=False)

+--------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+
|Min_Units_Sold|Max_Units_Sold|Min_Inventory|Max_Inventory|Min_Lead_Time|Max_Lead_Time|Min_Unit_Cost|Max_Unit_Cost|Min_Unit_Price|Max_Unit_Price|Min_Forecast|Max_Forecast|
+--------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+
|0             |59            |168          |990          |2            |14           |5.02         |19.76        |6.95          |35.1          |0.0         |61.42       |
+--------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+



In [13]:
# Verify that Promotion_Flag and Stockout_Flag contain only expected binary values.

inventory_df.groupBy("Promotion_Flag").count().orderBy("Promotion_Flag").show()

inventory_df.groupBy("Stockout_Flag").count().orderBy("Stockout_Flag").show()

+--------------+-----+
|Promotion_Flag|count|
+--------------+-----+
|             0|81980|
|             1| 9270|
+--------------+-----+

+-------------+-----+
|Stockout_Flag|count|
+-------------+-----+
|            0|91250|
+-------------+-----+



In [14]:
# Check how frequently different order quantities occur.
# This helps us understand whether most days have no replenishment order
# and whether large order quantities are rare but legitimate.

inventory_df.groupBy("Order_Quantity") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+--------------+-----+
|Order_Quantity|count|
+--------------+-----+
|0             |86223|
|386           |29   |
|218           |28   |
|426           |28   |
|413           |28   |
|440           |27   |
|429           |27   |
|209           |26   |
|257           |26   |
|453           |26   |
|225           |25   |
|201           |25   |
|302           |25   |
|255           |24   |
|478           |24   |
|301           |24   |
|382           |24   |
|488           |24   |
|427           |23   |
|408           |23   |
+--------------+-----+
only showing top 20 rows


In [15]:
# Count how many rows have an actual replenishment order.

inventory_df.select(
    F.sum(
        F.when(F.col("Order_Quantity") > 0, 1).otherwise(0)
    ).alias("Rows_With_Order"),

    F.sum(
        F.when(F.col("Order_Quantity") == 0, 1).otherwise(0)
    ).alias("Rows_Without_Order")
).show()

+---------------+------------------+
|Rows_With_Order|Rows_Without_Order|
+---------------+------------------+
|           5027|             86223|
+---------------+------------------+



In [16]:
# Identify records where selling price is lower than unit cost.
# These records could indicate negative unit margin or a data-quality concern.

inventory_df.filter(
    F.col("Unit_Price") < F.col("Unit_Cost")
).select(
    "SKU_ID",
    "Unit_Cost",
    "Unit_Price"
).show(20, truncate=False)


+------+---------+----------+
|SKU_ID|Unit_Cost|Unit_Price|
+------+---------+----------+
+------+---------+----------+



In [17]:
negative_margin_count = inventory_df.filter(
    F.col("Unit_Price") < F.col("Unit_Cost")
).count()

print("Rows where Unit_Price < Unit_Cost:", negative_margin_count)

Rows where Unit_Price < Unit_Cost: 0


In [18]:
# Calculate the first quartile (Q1) and third quartile (Q3) for Units_Sold.
# approxQuantile is efficient for large Spark datasets.

q1, q3 = inventory_df.approxQuantile(
    "Units_Sold",
    [0.25, 0.75],
    0.01
)

# Calculate the interquartile range.
iqr = q3 - q1

# Define the lower and upper outlier boundaries.
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 13.0
Q3: 27.0
IQR: 14.0
Lower bound: -8.0
Upper bound: 48.0


In [19]:
# Count records outside the IQR boundaries.

units_sold_outliers = inventory_df.filter(
    (F.col("Units_Sold") < lower_bound) |
    (F.col("Units_Sold") > upper_bound)
)

print("Potential Units_Sold outliers:", units_sold_outliers.count())

Potential Units_Sold outliers: 110


In [20]:
units_sold_outliers.select(
    "Date",
    "SKU_ID",
    "Warehouse_ID",
    "Units_Sold",
    "Promotion_Flag",
    "Demand_Forecast"
).orderBy(
    F.desc("Units_Sold")
).show(20, truncate=False)

+----------+------+------------+----------+--------------+---------------+
|Date      |SKU_ID|Warehouse_ID|Units_Sold|Promotion_Flag|Demand_Forecast|
+----------+------+------------+----------+--------------+---------------+
|2024-04-29|SKU_34|WH_4        |59        |1             |57.98          |
|2024-03-26|SKU_7 |WH_4        |58        |1             |59.43          |
|2024-04-23|SKU_38|WH_3        |58        |1             |56.12          |
|2024-04-14|SKU_36|WH_3        |57        |1             |54.67          |
|2024-04-12|SKU_3 |WH_1        |55        |1             |54.19          |
|2024-04-13|SKU_33|WH_4        |55        |1             |58.37          |
|2024-05-04|SKU_23|WH_3        |55        |1             |54.19          |
|2024-04-16|SKU_38|WH_3        |55        |1             |54.35          |
|2024-04-15|SKU_27|WH_4        |55        |1             |52.47          |
|2024-04-10|SKU_45|WH_5        |55        |1             |55.07          |
|2024-04-01|SKU_13|WH_2  

## Initial Data Exploration Findings

The supply chain dataset contains 91,250 records and 15 variables. The data
represents 365 dates, 50 SKUs, and 5 warehouses, producing a complete
Date × SKU × Warehouse structure.

The combination of Date, SKU_ID, and Warehouse_ID uniquely identifies each
record, confirming that the grain of the fact table is one SKU at one
warehouse on one date.

Initial data quality was strong. The numeric variables contained plausible
ranges, selling price was never below unit cost, and the binary promotion
indicator contained only expected values.

The Stockout_Flag contained only zero values, meaning that actual stockout
events cannot be analyzed directly using this field. Instead, later analysis
will create inventory-risk indicators based on inventory levels, reorder
points, demand, and supplier lead time.

Order_Quantity is highly zero-inflated. Most daily records do not contain a
replenishment order, while a smaller number contain relatively large order
quantities. These values should therefore not automatically be treated as
errors.

Using the IQR method, 110 Units_Sold observations were identified as potential
high-side outliers. Examination of these records showed that high sales were
commonly associated with promotions and similarly high demand forecasts.
Therefore, these observations appear to represent legitimate demand spikes
rather than data-quality errors.

The original observations will be preserved. Any outlier treatment used in
the preprocessing pipeline will create separate transformed features rather
than deleting valid business events.